# Ottawa zoning bylaw: EDA


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf
from IPython.display import Image, display

In [ ]:
DATA_DIR = Path("../data/ottawa")
BYLAW_PDF = DATA_DIR / "2026-050.pdf"  # rename to match your download

assert BYLAW_PDF.exists(), f"Download the bylaw PDF to {BYLAW_PDF.resolve()}"
pdf = pymupdf.open(BYLAW_PDF)

### Table of Contents + Text

In [ ]:
toc = pdf.get_toc()  # [[level, title, page_number], ...]
toc_df = pd.DataFrame(toc, columns=["level", "title", "page"])

content = []
for page_num, page in enumerate(pdf.pages()):
    content.append({"page": page_num + 1, "text": page.get_text()})
content = pd.DataFrame.from_records(content)

df = toc_df.merge(content, on="page", how="outer")

### Figures

In [ ]:
image_index = []
# TODO: get alt text on images

for page_num, page in enumerate(pdf.pages()):
    images = {"page": page_num + 1, "filenames": []}

    for _, img in enumerate(page.get_images(full=True)):
        xref = img[0]
        info = pdf.extract_image(xref)
        fname = f"p{page_num + 1}_x{xref}.{info['ext']}"
        path = (DATA_DIR / "output/figures" / fname).resolve()
        with open(path, "wb") as f:
            f.write(info["image"])  # save to disk
        images["filenames"].append(fname)
    image_index.append(images)
df = df.merge(pd.DataFrame(image_index), on="page", how="left")

In [ ]:
# temporary, remove all page=-1
df

In [ ]:
# many figures are missing level and titles, ffill from previous pages
df[["level", "title"]] = df[["level", "title"]].ffill()

# features
df["n_chars"] = df["text"].str.len()

### Chunking

In [ ]:
# TODO: may need to tune snap tolerance
# v1 - keep as big table
# TODO: get annotations
# for page in doc:
#     for annot in page.annots():
#         print(f'Annotation on page: {page.number} with type: {annot.type} and rect: {annot.rect}')

counter = 20
for i, page in enumerate(pdf[41:45]):
    print(f"page {i}")
    tables = page.find_tables(snap_x_tolerance=20)
    for table in tables:
        print(table.header)
        print(table.col_count)
        print(table.row_count)
        print(table.to_markdown())
    if i > counter:
        break


# TODO: need to detect tables that span multiple pages
# if first column has roman numerals, can potentially join,
# will keep detected tables on page as column
# add another column to dataframe of distinct tables which will span multiple pages
# some rows may also span multiple pages
# will need to copy over header, makes assumption that first row is always header


# section 199 definitions, each likely needs to be its own vector

In [ ]:
chunks = pdf.to_chunks(weights={"snap_x_tolerance": 10})  # ty: ignore[unresolved-attribute]
# save chunks

In [ ]:
pd.DataFrame(chunks.chunks).to_parquet("../data/ottawa/output/chunks.parquet")
pd.DataFrame(chunks.tables).to_parquet("../data/ottawa/output/tables.parquet")
pd.DataFrame(chunks.sections).to_parquet("../data/ottawa/output/sections.parquet")

In [ ]:
section_id = 100
print(chunks.sections[section_id])
child_chunk_ids = chunks.sections[section_id].child_chunk_ids
print(child_chunk_ids)
chunks.get(child_chunk_ids[0])

In [ ]:
# TODO: may need to tune snap tolerance
# v1 - keep as big table
# TODO: get annotations
# for page in doc:
#     for annot in page.annots():
#         print(f'Annotation on page: {page.number} with type: {annot.type} and rect: {annot.rect}')

counter = 20
for i, page in enumerate(pdf[41:45]):
    print(f"page {i}")
    tables = page.find_tables(snap_x_tolerance=10)
    for table in tables:
        print(table.header)
        print(table.col_count)
        print(table.row_count)
        print(table.to_markdown())
    if i > counter:
        break


# TODO: need to detect tables that span multiple pages
# if first column has roman numerals, can potentially join,
# will keep detected tables on page as column
# add another column to dataframe of distinct tables which will span multiple pages
# some rows may also span multiple pages
# will need to copy over header, makes assumption that first row is always header


# section 199 definitions, each likely needs to be its own vector

## quick validation 

In [ ]:
tokens = []
page_count = []
chars = []
chunk_id = []
for chunk in chunks.chunks:
    page_count.append(chunk.metadata.page_end - chunk.metadata.page_start)
    tokens.append(chunk.metadata.token_count)
    chars.append(len(chunk.text))
    chunk_id.append(chunk.id)

chunk_stats = pd.DataFrame(
    {"chunk_id": chunk_id, "tokens": tokens, "page_count": page_count, "n_chars": chars}
)

In [ ]:
chunk_stats["n_chars"].hist()

In [ ]:
chunk_stats["tokens"].hist()


**771** (page 409, actually page 406)
this chunk only contains a heading?
there is more text and paragraphs on this page, were they captured in chunks following 771?

**20**, (page 17, actual 14),
small section from this page, seems to get path right 

**633** - seems ok

**748** - table columns are split, need to use same params as above test
```||**Table 1311-2 – VM2 Su**|**bzone Provisions**|```


**498** picks up on italics with _ 

**224** good example of nested bullet points, should all be in one chunk which is good

**198**

**760** 
  
**169** figure with caption 

**88** definitions section, says section E but actually section F, however path for definitions kind of irrelevant. interesting how each has french translation in brackets following definition, doesnt appear in pdf. should all be individual chunks  
types ['paragraph']
section path ['Section 199 – Definitions', '<mark>E</mark>']
**Façade** means all exterior wall surfaces facing a lot line that abuts a public street, and includes any doors, garage doors or windows set into such walls, as well as any articulation between such walls.
( _façade_ )
**Flood-Proofed** means those established technical measures for buildings, structures or properties prone to flooding, which are implemented to reduce or eliminate flood damage.
Flood proofed non-residential uses and alterations to existing non-conforming uses may incorporate wet passive measures.
( _protégé contre les inondations_ )

**685**

**953** large table missing key column names because they dont appear on the page 

**3** good at linking figure with correct caption, -1 text at top of page, but most of the page refers to -2
   

In [ ]:
# sample chunks into df and display pages where they appear
# apply label
np.random.seed(10)
tests = np.random.rand(20) * 1000
tests = tests.astype(int)
print(tests)

for i, test in enumerate(tests):
    print("--------------------")
    print(f"TEST {i}")

    chunk = chunks.chunks[test]
    print(f"pages ({chunk.metadata.page_start}, {chunk.metadata.page_end})")
    print(f"types {chunk.metadata.types}")
    print(f"section path {chunk.metadata.section_path}")
    print(chunk.text)
    print(chunk)

    display(Image(pdf[chunk.metadata.page_start - 1].get_pixmap(dpi=100).tobytes("png")))

In [ ]:
# TODO: may need to tune snap tolerance
# v1 - keep as big table
# TODO: get annotations
# for page in doc:
#     for annot in page.annots():
#         print(f'Annotation on page: {page.number} with type: {annot.type} and rect: {annot.rect}')

counter = 20
for i, page in enumerate(pdf[41:45]):
    print(f"page {i}")
    tables = page.find_tables(snap_x_tolerance=10)
    for table in tables:
        print(table.header)
        print(table.col_count)
        print(table.row_count)
        print(table.to_markdown())
    if i > counter:
        break


# TODO: need to detect tables that span multiple pages
# if first column has roman numerals, can potentially join,
# will keep detected tables on page as column
# add another column to dataframe of distinct tables which will span multiple pages
# some rows may also span multiple pages
# will need to copy over header, makes assumption that first row is always header


# section 199 definitions, each likely needs to be its own vector

In [ ]:
### OCR vs no OCR
# find pages with little text and compare chunk with ocr vs without

In [ ]:
# compare total number of chars from using get_text to total chars across chunks

In [ ]:
# find pages with no chunks

In [ ]:
# Check that every Section number in the table of contents shows up in some chunk’s
#  text or metadata.

### Boundaries
<!-- Check the share of chunks that start with a heading or a provision number like (1) or (a). A chunk starting mid-sentence (lowercase first word, no heading) means a bad split.
Check the share of chunks that contain more than one “Section ###” heading, which means sections were merged.
Look at the shortest and longest 20 chunks. The extremes are where the bugs live. -->

<!-- Duplication and noise

Look for exact and near duplicates, such as repeated headers and footers or tables captured twice (once as a table, once as text).
Count leftover page furniture: page numbers, “Zoning By-law 2026-50” running headers, hyphenated line breaks like By-\nlaw. -->

## Docling

In [ ]:
from docling.chunking import HybridChunker
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

BYLAW_PDF = Path("../data/ottawa/2026-050.pdf")

# 1. Convert: no OCR (text layer is good), with table structure detection on
opts = PdfPipelineOptions(do_ocr=False, do_table_structure=True)
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
)
result = converter.convert(BYLAW_PDF, page_range=(1, 500))  # start small; drop page_range for all
doc = result.document

doc.save_as_json(
    Path("../data/ottawa/output/docling_400-420.json")
)  # cache, so you don't reconvert

# 2. Chunk by heading hierarchy
chunker = HybridChunker()  # splits/merges using a tokenizer; defaults are fine to start
rows = []
for i, chunk in enumerate(chunker.chunk(dl_doc=doc)):
    pages = sorted({p.page_no for item in chunk.meta.doc_items for p in item.prov})
    rows.append(
        {
            "chunk_id": i,
            "headings": " > ".join(chunk.meta.headings or []),
            "kinds": sorted(
                {item.label for item in chunk.meta.doc_items}
            ),  # text, table, list_item...
            "page_start": pages[0] if pages else None,
            "page_end": pages[-1] if pages else None,
            "text": chunk.text,
            "text_for_embedding": chunker.contextualize(chunk),  # headings + text
        }
    )

chunks_df = pd.DataFrame(rows)
chunks_df.head()

In [ ]:
# chunks_df[chunks_df['headings']=='Urban Exceptions 301-400']
chunks_df.head(20)